# `relay.qnn.requantize`

In [ ]:
import tvm
from tvm import te
import numpy as np
from tvm import relay
from tvm.contrib import graph_executor

# 定义舍入模式：向上舍入和四舍五入
roundings = ["UPWARD", "TONEAREST"]
# 定义计算数据类型
compute_dtypes = ["float32", "float64", "int64"]
# 定义输出量化数据类型
out_dtypes = ["int8", "int16"]


def verify(mod, goldens, target="llvm"):
    """
    验证QNN重量化模型的输出是否符合预期

    参数:
    mod: 要验证的Relay模块
    goldens: 包含输入数据和预期输出的元组
    target: 目标设备，默认为"llvm"
    """
    with tvm.transform.PassContext(opt_level=3):
        # 构建模型
        graph, lib, params = relay.build(mod, target, params=None)
        golden_data, golden_output = goldens
        # 创建执行器
        rt_mod = graph_executor.create(graph, lib, device=tvm.cpu(0))
        # 设置输入数据
        rt_mod.set_input("input_data", golden_data)
        rt_mod.set_input(**params)
        # 运行模型
        rt_mod.run()
        # 获取输出
        res = rt_mod.get_output(0).numpy()
        # 验证结果
        np.testing.assert_equal(res, golden_output)


def get_mod(
    data_shape,
    data_dtype,
    out_dtype,
    input_scale,
    output_scale,
    input_zero_point=0,
    output_zero_point=0,
    rounding="None",
    compute_dtype="None",
    axis=0,
):
    """
    创建QNN重量化模型的辅助函数

    参数:
    data_shape: 输入数据形状
    data_dtype: 输入数据类型
    out_dtype: 输出数据类型
    input_scale: 输入缩放比例
    output_scale: 输出缩放比例
    input_zero_point: 输入零点偏移，默认为0
    output_zero_point: 输出零点偏移，默认为0
    rounding: 舍入模式，默认为"None"
    compute_dtype: 计算数据类型，默认为"None"
    axis: 通道轴，默认为0

    返回:
    创建的Relay模块
    """
    # 创建输入变量
    input_data = relay.var("input_data", shape=data_shape, dtype=data_dtype)
    # 处理输入缩放比例
    if isinstance(input_scale, float):
        input_scale_expr = relay.const(input_scale, "float32")
    else:
        input_scale_expr = relay.const(np.array(input_scale).astype("float32"))

    # 处理输入零点偏移
    if isinstance(input_zero_point, float):
        input_zero_point_expr = relay.const(input_zero_point, "int32")
    else:
        input_zero_point_expr = relay.const(np.array(input_zero_point).astype("int32"))

    # 创建重量化操作
    mod = relay.qnn.requantize(
        input_data,
        input_scale=input_scale_expr,
        input_zero_point=input_zero_point_expr,
        output_scale=relay.const(output_scale, "float32"),
        output_zero_point=relay.const(output_zero_point, "int32"),
        axis=axis,
        rounding=rounding,
        compute_dtype=compute_dtype,
        out_dtype=out_dtype,
    )

    # 构建函数和模块
    mod = relay.Function(relay.analysis.free_vars(mod), mod)
    mod = tvm.IRModule.from_expr(mod)
    return mod


def test_same_scale():
    """
    测试输入和输出缩放比例相同的情况

    当输入和输出缩放比例相同时，重量化操作应该是一个恒等变换
    """
    # 生成测试数据：范围在[-100, 100)的整数
    golden_data = np.arange(-100, 100, 1).astype("int32")
    # 预期输出与输入相同
    golden_output = golden_data
    # 测试所有计算数据类型、舍入模式和输出数据类型的组合
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型
                mod = get_mod(
                    data_shape=(200,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=0.5,
                    output_scale=0.5,  # 相同的缩放比例
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )
                # 确保没有右移操作（因为缩放比例相同）
                assert "right_shift" not in mod.astext()
                # 验证结果
                verify(mod, (golden_data, golden_output))


def test_scalar_same_scale():
    """
    测试标量输入且缩放比例相同的情况

    验证对标量数据进行重量化时的正确性
    """
    # 生成标量测试数据
    golden_data = np.array(-10).astype("int32")
    # 预期输出与输入相同
    golden_output = golden_data
    # 测试所有计算数据类型、舍入模式和输出数据类型的组合
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型（标量输入）
                mod = get_mod(
                    data_shape=(),  # 标量形状
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=0.5,
                    output_scale=0.5,  # 相同的缩放比例
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )
                # 确保没有右移操作
                assert "right_shift" not in mod.astext()
                # 验证结果
                verify(mod, (golden_data, golden_output))


def test_downscale():
    """
    测试输出缩放比例大于输入缩放比例（缩小）的情况

    验证当输出缩放比例大于输入缩放比例时，重量化操作的正确性
    """
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型（缩放比例 1->16）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=1,
                    output_scale=16,  # 输出缩放比例更大（缩小）
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )

                # 测试正值
                # 8对应0.5，结果为1
                golden_data = np.arange(0, 32, 1).astype("int32")
                golden_output = np.repeat([0, 1, 2], [8, 16, 8])
                verify(mod, (golden_data, golden_output))

                # 测试负值
                # -8对应-0.5。对于UPWARD舍入，结果为0
                golden_data = np.arange(0, -32, -1).astype("int32")
                if rounding == "UPWARD":
                    golden_output = np.repeat([0, -1, -2], [9, 16, 7])
                else:
                    golden_output = np.repeat([0, -1, -2], [8, 16, 8])
                verify(mod, (golden_data, golden_output))

                # 测试不同的缩放比例（1->4）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=1,
                    output_scale=4,
                    rounding=rounding,
                )

                # 测试正值
                # 2I对应0.5，结果为1
                golden_data = np.arange(0, 32, 1).astype("int32")
                golden_output = np.repeat([0, 1, 2, 3, 4, 5, 6, 7, 8], [2, 4, 4, 4, 4, 4, 4, 4, 2])
                verify(mod, (golden_data, golden_output))

                # 测试负值
                # -8对应-0.5。对于UPWARD舍入，结果为0
                golden_data = np.arange(0, -32, -1).astype("int32")
                if rounding == "UPWARD":
                    golden_output = np.repeat(
                        [0, -1, -2, -3, -4, -5, -6, -7, -8], [3, 4, 4, 4, 4, 4, 4, 4, 1]
                    )
                else:
                    golden_output = np.repeat(
                        [0, -1, -2, -3, -4, -5, -6, -7, -8], [2, 4, 4, 4, 4, 4, 4, 4, 2]
                    )
                verify(mod, (golden_data, golden_output))

            # 测试uint8输出数据类型
            mod = get_mod(
                data_shape=(32,),
                data_dtype="int32",
                out_dtype="uint8",
                input_scale=1,
                output_scale=16,
                rounding=rounding,
            )

            # 测试正值
            golden_data = np.arange(0, 32, 1).astype("int32")
            golden_output = np.repeat([0, 1, 2], [8, 16, 8])
            verify(mod, (golden_data, golden_output))

            # 测试uint8输入和输出数据类型
            mod = get_mod(
                data_shape=(32,),
                data_dtype="uint8",
                out_dtype="uint8",
                input_scale=1,
                output_scale=16,
                rounding=rounding,
            )

            # 测试正值
            golden_data = np.arange(0, 32, 1).astype("int32")
            golden_output = np.repeat([0, 1, 2], [8, 16, 8])
            verify(mod, (golden_data, golden_output))


def test_upscale():
    """
    测试输出缩放比例小于输入缩放比例（放大）的情况

    验证当输出缩放比例小于输入缩放比例时，重量化操作的正确性
    """
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型（缩放比例 2->1）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=2,
                    output_scale=1,  # 输出缩放比例更小（放大）
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )

                # 测试正值
                golden_data = np.arange(0, 32, 1).astype("int32")
                # 由于输出缩放比例是输入的一半，输出值应该是输入值的2倍
                golden_output = np.multiply(2, golden_data)
                verify(mod, (golden_data, golden_output))

                # 测试负值
                golden_data = np.arange(0, -32, -1).astype("int32")
                golden_output = np.multiply(2, golden_data)
                verify(mod, (golden_data, golden_output))


def test_non_power_of_two():
    """
    测试非2的幂次缩放比例的情况

    验证当缩放比例不是2的幂次时，重量化操作的正确性
    """
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型（缩放比例 1->3）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=1,
                    output_scale=3,  # 非2的幂次缩放比例
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )

                # 测试正值
                # 输入数据乘以3，以便重量化后得到原始范围
                golden_data = np.multiply(np.arange(0, 32, 1).astype("int32"), 3)
                golden_output = np.arange(0, 32, 1)
                verify(mod, (golden_data, golden_output))

                # 测试负值
                golden_data = np.multiply(np.arange(0, -32, -1).astype("int32"), 3)
                golden_output = np.arange(0, -32, -1)
                verify(mod, (golden_data, golden_output))

                # 测试不同的缩放比例（3->1）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=3,
                    output_scale=1,
                    rounding=rounding,
                )

                # 测试正值
                golden_data = np.arange(0, 32, 1).astype("int32")
                # 由于输出缩放比例是输入的1/3，输出值应该是输入值的3倍
                golden_output = np.multiply(golden_data, 3)
                verify(mod, (golden_data, golden_output))

                # 测试负值
                golden_data = np.arange(0, -32, -1).astype("int32")
                golden_output = np.multiply(golden_data, 3)
                verify(mod, (golden_data, golden_output))


def test_saturation_int8():
    """
    测试int8数据类型的饱和情况

    验证当重量化结果超出int8类型范围时的饱和行为
    """
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            # 创建重量化模型（输出类型为int8）
            mod = get_mod(
                data_shape=(16,),
                data_dtype="int32",
                out_dtype="int8",
                input_scale=0.5,
                output_scale=0.5,
                rounding=rounding,
                compute_dtype=compute_dtype,
            )
            # 测试正值饱和
            golden_data = np.arange(0, 16, 1).astype("int32")
            golden_data = np.add(120, golden_data)  # 从120开始，超出int8的最大值127
            output = np.array(
                [120, 121, 122, 123, 124, 125, 126, 127, 127, 127, 127, 127, 127, 127, 127, 127]
            )
            golden_output = output
            verify(mod, (golden_data, golden_output))

            # 测试负值饱和
            golden_data = np.arange(0, -16, -1).astype("int32")
            golden_data = np.add(-120, golden_data)  # 从-120开始，超出int8的最小值-128
            output = np.array(
                [
                    -120,
                    -121,
                    -122,
                    -123,
                    -124,
                    -125,
                    -126,
                    -127,
                    -128,
                    -128,
                    -128,
                    -128,
                    -128,
                    -128,
                    -128,
                    -128,
                ]
            )
            golden_output = output
            verify(mod, (golden_data, golden_output))


def test_saturation_int16():
    """
    测试int16数据类型的饱和情况

    验证当重量化结果超出int16类型范围时的饱和行为
    """
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            # 创建重量化模型（输出类型为int16）
            mod = get_mod(
                data_shape=(16,),
                data_dtype="int32",
                out_dtype="int16",
                input_scale=0.5,
                output_scale=0.5,
                rounding=rounding,
                compute_dtype=compute_dtype,
            )
            # 测试正值饱和
            golden_data = np.arange(0, 16, 1).astype("int32")
            golden_data = np.add(32760, golden_data)  # 从32760开始，超出int16的最大值32767
            output = np.array(
                [
                    32760,
                    32761,
                    32762,
                    32763,
                    32764,
                    32765,
                    32766,
                    32767,
                    32767,
                    32767,
                    32767,
                    32767,
                    32767,
                    32767,
                    32767,
                    32767,
                ]
            )
            golden_output = output
            verify(mod, (golden_data, golden_output))

            # 测试负值饱和
            golden_data = np.arange(0, -16, -1).astype("int32")
            golden_data = np.add(-32760, golden_data)  # 从-32760开始，超出int16的最小值-32768
            output = np.array(
                [
                    -32760,
                    -32761,
                    -32762,
                    -32763,
                    -32764,
                    -32765,
                    -32766,
                    -32767,
                    -32768,
                    -32768,
                    -32768,
                    -32768,
                    -32768,
                    -32768,
                    -32768,
                    -32768,
                ]
            )
            golden_output = output
            verify(mod, (golden_data, golden_output))


def test_zero_point():
    """
    测试零点偏移对重量化的影响

    分别测试输出零点偏移和输入零点偏移的情况
    """
    # 测试输出零点偏移
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            # 创建重量化模型（输出零点偏移为1）
            mod = get_mod(
                data_shape=(32,),
                data_dtype="int32",
                out_dtype="int8",
                input_scale=1,
                output_scale=16,
                output_zero_point=1,  # 输出零点偏移
                rounding=rounding,
                compute_dtype=compute_dtype,
            )

            # 测试正值
            golden_data = np.arange(0, 32, 1).astype("int32")
            golden_output = np.repeat([0, 1, 2], [8, 16, 8])
            golden_output = np.add(1, golden_output)  # 加上输出零点偏移
            verify(mod, (golden_data, golden_output))

            # 测试负值
            golden_data = np.arange(-32, -64, -1).astype("int32")
            if rounding == "UPWARD":
                golden_output = np.repeat([-2, -3, -4], [9, 16, 7])
            else:
                golden_output = np.repeat([-2, -3, -4], [8, 16, 8])
            golden_output = np.add(1, golden_output)  # 加上输出零点偏移
            verify(mod, (golden_data, golden_output))

    # 测试输入零点偏移
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型（输入零点偏移为16）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=1,
                    output_scale=16,
                    input_zero_point=16,  # 输入零点偏移
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )

                # 测试正值
                golden_data = np.arange(32, 64, 1).astype("int32")
                golden_output = np.repeat([2, 3, 4], [8, 16, 8])
                golden_output = np.subtract(golden_output, 1)  # 减去输入零点偏移的影响
                verify(mod, (golden_data, golden_output))

                # 测试负值
                golden_data = np.arange(-32, -64, -1).astype("int32")
                if rounding == "UPWARD":
                    golden_output = np.repeat([-2, -3, -4], [9, 16, 7])
                else:
                    golden_output = np.repeat([-2, -3, -4], [8, 16, 8])
                golden_output = np.subtract(golden_output, 1)  # 减去输入零点偏移的影响
                verify(mod, (golden_data, golden_output))


def test_per_channel_same_scale():
    """
    测试通道级相同缩放比例的情况

    验证当每个通道使用相同的缩放比例时，重量化操作的正确性
    """
    # 测试2D数据
    golden_data = np.arange(-5, 5, 1).astype("int32").reshape((5, 2))
    golden_output = golden_data  # 预期输出与输入相同
    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            for qnn_out_dtype in out_dtypes:
                # 创建重量化模型（每个通道使用相同的缩放比例）
                mod = get_mod(
                    data_shape=(5, 2),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=[0.5, 0.5],  # 通道级缩放比例
                    output_scale=0.5,
                    axis=1,  # 通道轴
                    rounding=rounding,
                    compute_dtype=compute_dtype,
                )
                verify(mod, (golden_data, golden_output))

    # 测试3D数据（改变通道轴）
    golden_data = np.arange(-10, 10, 1).astype("int32").reshape((2, 2, 5))
    golden_output = golden_data

    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            # 创建重量化模型（不同的输入形状和通道轴）
            mod = get_mod(
                data_shape=(2, 2, 5),
                data_dtype="int32",
                out_dtype="int8",
                input_scale=[0.5, 0.5],  # 通道级缩放比例
                output_scale=0.5,
                axis=1,  # 通道轴
                rounding=rounding,
                compute_dtype=compute_dtype,
            )
            verify(mod, (golden_data, golden_output))


def test_per_channel_different_scale():
    """
    测试通道级不同缩放比例的情况

    验证当每个通道使用不同的缩放比例时，重量化操作的正确性
    """
    # 测试2D数据（不同通道缩放比例）
    golden_data = np.arange(-5, 5, 1).astype("int32").reshape((5, 2))
    golden_output = np.array([-5, -2, -3, -1, -1, 0, 1, 1, 3, 2]).reshape((5, 2))

    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            # 创建重量化模型（每个通道使用不同的缩放比例）
            mod = get_mod(
                data_shape=(5, 2),
                data_dtype="int32",
                out_dtype="int8",
                input_scale=[0.5, 0.25],  # 不同的通道缩放比例
                output_scale=0.5,
                axis=1,
                rounding=rounding,
                compute_dtype=compute_dtype,
            )
            verify(mod, (golden_data, golden_output))

    # 测试3D数据（改变通道轴）
    golden_data = np.arange(-20, 20, 2).astype("int32").reshape((2, 2, 5))
    golden_output = np.array(
        [-20, -18, -16, -14, -12, -5, -4, -3, -2, -1, 0, 2, 4, 6, 8, 5, 6, 7, 8, 9]
    ).reshape((2, 2, 5))

    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            mod = get_mod(
                data_shape=(2, 2, 5),
                data_dtype="int32",
                out_dtype="int8",
                input_scale=[0.5, 0.25],
                output_scale=0.5,
                axis=1,
                rounding=rounding,
                compute_dtype=compute_dtype,
            )
            verify(mod, (golden_data, golden_output))

    # 测试输入缩放比例大于输出缩放比例的情况
    golden_data = np.arange(-5, 5, 1).astype("int32").reshape((5, 2))
    golden_output = np.array([-10, -2, -6, -1, -2, 0, 2, 1, 6, 2]).reshape((5, 2))

    for compute_dtype in compute_dtypes:
        for rounding in roundings:
            mod = get_mod(
                data_shape=(5, 2),
                data_dtype="int32",
                out_dtype="int8",
                input_scale=[1.0, 0.25],  # 输入缩放比例大于输出
                output_scale=0.5,
                axis=1,
                rounding=rounding,
                compute_dtype=compute_dtype,
            )
            verify(mod, (golden_data, golden_output))


def test_default_cfg_and_no_args():
    """
    测试默认配置且无参数的情况

    验证使用默认配置且不指定额外参数时，重量化操作的正确性
    """
    for qnn_out_dtype in out_dtypes:
        # 创建重量化模型（使用默认配置）
        mod = get_mod(
            data_shape=(32,),
            data_dtype="int32",
            out_dtype=qnn_out_dtype,
            input_scale=1,
            output_scale=16,
            # 不指定舍入模式等参数，使用默认值
        )
        golden_data = np.arange(0, -32, -1).astype("int32")
        golden_output = np.repeat([0, -1, -2], [9, 16, 7])
        verify(mod, (golden_data, golden_output))


def test_non_default_cfg_and_no_args():
    """
    测试非默认配置且无参数的情况

    验证使用非默认全局配置且不指定额外参数时，重量化操作的正确性
    """
    for rounding_cfg in roundings:
        for qnn_out_dtype in out_dtypes:
            # 设置非默认全局配置
            with relay.qnn.requantize_config(rounding=rounding_cfg):
                # 创建重量化模型（不指定额外参数）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=1,
                    output_scale=16,
                )

                golden_data = np.arange(0, -32, -1).astype("int32")

                # 根据全局配置的舍入模式设置预期输出
                if rounding_cfg == "UPWARD":
                    golden_output = np.repeat([0, -1, -2], [9, 16, 7])
                else:
                    golden_output = np.repeat([0, -1, -2], [8, 16, 8])
                verify(mod, (golden_data, golden_output))


def test_default_cfg_and_args():
    """
    测试默认配置且有参数的情况

    验证使用默认全局配置但指定局部参数时，重量化操作的正确性
    """
    for rounding in roundings:
        for qnn_out_dtype in out_dtypes:
            # 设置默认全局配置
            with relay.qnn.requantize_config(rounding="UPWARD"):
                # 创建重量化模型（指定局部参数，覆盖全局配置）
                mod = get_mod(
                    data_shape=(32,),
                    data_dtype="int32",
                    out_dtype=qnn_out_dtype,
                    input_scale=1,
                    output_scale=16,
                    rounding=rounding,  # 局部参数覆盖全局配置
                )

                golden_data = np.arange(0, -32, -1).astype("int32")

                # 根据局部参数的舍入模式设置预期输出
                if rounding == "UPWARD":
                    golden_output = np.repeat([0, -1, -2], [9, 16, 7])
                else:
                    golden_output = np.repeat([0, -1, -2], [8, 16, 8])
                verify(mod, (golden_data, golden_output))


def test_non_default_cfg_and_args():
    """
    测试非默认配置且有参数的情况

    验证使用非默认全局配置且指定局部参数时，重量化操作的正确性
    """
    for rounding_arg in roundings:
        for rounding_cfg in roundings:
            for qnn_out_dtype in out_dtypes:
                # 设置非默认全局配置
                with relay.qnn.requantize_config(rounding=rounding_cfg):
                    # 创建重量化模型（指定局部参数，覆盖全局配置）
                    mod = get_mod(
                        data_shape=(32,),
                        data_dtype="int32",
                        out_dtype=qnn_out_dtype,
                        input_scale=1,
                        output_scale=16,
                        rounding=rounding_arg,  # 局部参数覆盖全局配置
                    )

                    golden_data = np.arange(0, -32, -1).astype("int32")

                    # 根据局部参数的舍入模式设置预期输出
                    if rounding_arg == "UPWARD":
                        golden_output = np.repeat([0, -1, -2], [9, 16, 7])
                    else:
                        golden_output = np.repeat([0, -1, -2], [8, 16, 8])
                    verify(mod, (golden_data, golden_output))


if __name__ == "__main__":
    # 执行所有测试函数
    test_same_scale()  # 测试相同缩放比例
    test_scalar_same_scale()  # 测试标量相同缩放比例
    test_downscale()  # 测试缩小缩放比例
    test_upscale()  # 测试放大缩放比例
    test_non_power_of_two()  # 测试非2的幂次缩放比例
    test_saturation_int8()  # 测试int8饱和情况
    test_saturation_int16()  # 测试int16饱和情况
    test_zero_point()  # 测试零点偏移
    test_per_channel_same_scale()  # 测试通道级相同缩放比例
    test_per_channel_different_scale()  # 测试通道级不同缩放比例
    test_default_cfg_and_no_args()  # 测试默认配置且无参数
    test_non_default_cfg_and_no_args()  # 测试非默认配置且无参数
    test_default_cfg_and_args()  # 测试默认配置且有参数
    test_non_default_cfg_and_args()  # 测试非默认配置且有参数
